In [ ]:
# ============================================================
# SETUP - Run this cell first
# ============================================================
!git clone https://github.com/tatipar/temporalgnn-nids.git
import sys
sys.path.append('/content/temporalgnn-nids/code/python')

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/nids-mitre/')

In [ ]:
!pip install torch_geometric -q

In [ ]:
import os
import gc
import glob
import pickle
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

In [ ]:
from utils.datasets import NF_IDS_Dataset
from utils.models   import (
    EdgeGRU_Baseline_NoX,
    EdgeGRU_Baseline_Entropy,
    E_GraphSAGE,
    StaticGNN_Identity,     StaticGNN_Identity_Entropy,
    ST_GNN_Identity,        ST_GNN_Identity_Entropy,
)
from utils.training    import evaluate
from utils.evaluation  import gather_metrics, apply_1sd_rule
from utils.mitre       import (
    extract_mitre_events, count_total_flows,
    metrics_per_tactic, analyze_timeline,
    analyze_early_warning, analyze_all_lateral_movements_pairs,
)
from utils.metrics import calculate_full_temporal_metrics, calculate_mttd_metrics


In [ ]:
# ── Directories ────────────────────────────────────────────────────────────
DIR_NOENT  = './results_earlystopping'
DIR_ENT    = './results_earlystopping_entropy'
MITRE_DIR  = './analysis_mitre_comparison'
PLOTS_DIR  = './analysis_mitre_comparison/plots'
os.makedirs(MITRE_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# ── Shared model hyperparameters ───────────────────────────────────────────
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_config = {
    'model_name': None, 'type': None,
    'model_params': {
        'node_dim': 16, 'edge_dim': 32, 'hidden_dim': 32,
        'dropout': 0.2, 'output_bias_init': -2.9968,
    },
    'extra_params': {
        'epochs': 60, 'batch_steps': 10,
        'pos_weight': 2.0, 'learning_rate': 0.005,
    }
}

# ── Network context (attack infrastructure) ────────────────────────────────
C2_SERVERS_DAY1 = ['66.111.57.61', '94.182.163.51']
EXTERNAL_ATTACKERS_DAY1 = [
    '172.229.222.45', '66.198.178.50', '52.50.201.26', '65.52.108.213',
    '52.6.193.105',   '23.239.24.18',  '66.235.146.78','131.202.1.45',
    '52.85.131.173',  '37.230.149.51'
]
C2_SERVERS_DAY2 = ['69.16.232.48', '117.198.97.175', '79.18.65.124']
EXTERNAL_ATTACKERS_DAY2 = [
    '54.85.40.222',   '183.220.102.254', '65.52.108.196',
    '152.163.50.3',   '52.229.173.178',  '183.83.254.235', '52.20.73.160'
]
INTERNAL_NET = '172.31.0.0/16'

print(f'Device: {DEVICE}')

In [ ]:
# ── Champion selection ─────────────────────────────────────────────────────
NAME_MAP_NOENT = {
    'EGraphSAGE_BiasOn':                   'E-GraphSAGE',
    'EdgeGRU_NoX_BiasOn':                  'Edge GRU',
    'StaticGNN_BiasOn_robust_Identity':    'Static GNN',
    'ST_GNN_BiasOn_robust_Identity_clone': 'ST-GNN',
}
NAME_MAP_ENT = {
    'EdgeGRU_BiasOn_entropy':                       'Edge GRU + Entropy',
    'StaticGNN_BiasOn_robust_Identity_entropy':     'Static GNN + Entropy',
    'ST_GNN_BiasOn_robust_Identity_clone_entropy':  'ST-GNN + Entropy',
}

print('── Noentropy ──')
df_metrics_noent = gather_metrics(os.path.join(DIR_NOENT, 'logs'), NAME_MAP_NOENT)
df_champ_noent   = apply_1sd_rule(df_metrics_noent)

print('\n── Entropy ──')
df_metrics_ent   = gather_metrics(os.path.join(DIR_ENT, 'logs'), NAME_MAP_ENT)
df_champ_ent     = apply_1sd_rule(df_metrics_ent)

In [ ]:
# ── Helper: load champion model + threshold ────────────────────────────────
def load_champion(model_class, experiment_name, df_champions, results_dir, device):
    """
    Load the champion .pth and its optimal threshold from the .npz.
    Returns (model, threshold, seed).
    """
    champ = df_champions[df_champions['Raw_Dir_Name'] == experiment_name]
    if champ.empty:
        raise ValueError(f'No champion found for {experiment_name}')

    seed = int(champ['Seed'].iloc[0])

    npz_path  = f'{results_dir}/logs/{experiment_name}/thresholds_{experiment_name}.npz'
    threshold = float(np.load(npz_path)[f'seed_{seed}'])

    csv_files = glob.glob(f'{results_dir}/logs/{experiment_name}/run_metrics_{experiment_name}.csv')
    if not csv_files:
        csv_files = glob.glob(f'{results_dir}/logs/{experiment_name}/metrics_newth_{experiment_name}.csv')
    df_csv = pd.read_csv(csv_files[0])
    df_csv.columns = df_csv.columns.str.lower()
    run_id = df_csv[df_csv['model_name'].str.contains(f'seed{seed}')]['run_id'].iloc[0]

    pth_files = glob.glob(f'{results_dir}/saved_models/{experiment_name}/{run_id}_*.pth')
    if not pth_files:
        raise FileNotFoundError(f'No .pth found for run_id={run_id}')

    model = model_class(**model_config['model_params']).to(device)
    model.load_state_dict(torch.load(pth_files[0], map_location=device))
    model.eval()

    print(f'  Loaded: {experiment_name} | seed={seed} | threshold={threshold:.6f}')
    return model, threshold, seed

# Section 1 — Extract MITRE Events (Inference)

In [ ]:
# ── Datasets ───────────────────────────────────────────────────────────────
test1_noent = NF_IDS_Dataset(root_dir='./dataset_processed',                  split='test')
test1_ent   = NF_IDS_Dataset(root_dir='./dataset_processed_entropy',          split='test')
test2_noent = NF_IDS_Dataset(root_dir='./dataset_processed_thu0103',          split='test2')
test2_ent   = NF_IDS_Dataset(root_dir='./dataset_processed_thu0103_entropy',  split='test2')

loader_t1_noent = DataLoader(test1_noent, batch_size=1, shuffle=False, num_workers=0)
loader_t1_ent   = DataLoader(test1_ent,   batch_size=1, shuffle=False, num_workers=0)
loader_t2_noent = DataLoader(test2_noent, batch_size=1, shuffle=False, num_workers=0)
loader_t2_ent   = DataLoader(test2_ent,   batch_size=1, shuffle=False, num_workers=0)

# ── IP maps (shared across entropy/noentropy — same graph topology) ────────
with open('./dataset_processed/ip_map_day1.pkl', 'rb') as f:
    ip_to_id_day1 = pickle.load(f)
id_to_ip_day1 = {v: k for k, v in ip_to_id_day1.items()}

with open('./dataset_processed_thu0103/ip_map_day2.pkl', 'rb') as f:
    ip_to_id_day2 = pickle.load(f)
id_to_ip_day2 = {v: k for k, v in ip_to_id_day2.items()}

print('Datasets and IP maps loaded.')

In [ ]:
# ── ST-GNN + Entropy: reuse existing CSVs ─────────────────────────────────
shutil.copy('./analysis4paper_entropy/mitre_test1.csv',
            f'{MITRE_DIR}/mitre_test1_stgnn_ent.csv')
shutil.copy('./analysis4paper_entropy/mitre_test2.csv',
            f'{MITRE_DIR}/mitre_test2_stgnn_ent.csv')
print('ST-GNN + Entropy: CSVs copied from analysis4paper_entropy.')

In [ ]:
# ── EdgeGRU + Entropy ──────────────────────────────────────────────────────
print('Running EdgeGRU + Entropy...')
model, threshold, _ = load_champion(
    EdgeGRU_Baseline_Entropy, 'EdgeGRU_BiasOn_entropy', df_champ_ent, DIR_ENT, DEVICE
)

df = extract_mitre_events(loader_t1_ent, DEVICE, model, threshold, True, id_to_ip_day1,
                          C2_SERVERS_DAY1, EXTERNAL_ATTACKERS_DAY1, INTERNAL_NET)
df.to_csv(f'{MITRE_DIR}/mitre_test1_egru_ent.csv', index=False)

df = extract_mitre_events(loader_t2_ent, DEVICE, model, threshold, True, id_to_ip_day2,
                          C2_SERVERS_DAY2, EXTERNAL_ATTACKERS_DAY2, INTERNAL_NET)
df.to_csv(f'{MITRE_DIR}/mitre_test2_egru_ent.csv', index=False)

del model; gc.collect(); torch.cuda.empty_cache()
print('Done.')

In [ ]:
# ── Static GNN (noentropy) ─────────────────────────────────────────────────
print('Running Static GNN...')
model, threshold, _ = load_champion(
    StaticGNN_Identity, 'StaticGNN_BiasOn_robust_Identity', df_champ_noent, DIR_NOENT, DEVICE
)

df = extract_mitre_events(loader_t1_noent, DEVICE, model, threshold, False, id_to_ip_day1,
                          C2_SERVERS_DAY1, EXTERNAL_ATTACKERS_DAY1, INTERNAL_NET)
df.to_csv(f'{MITRE_DIR}/mitre_test1_sgnn_noent.csv', index=False)

df = extract_mitre_events(loader_t2_noent, DEVICE, model, threshold, False, id_to_ip_day2,
                          C2_SERVERS_DAY2, EXTERNAL_ATTACKERS_DAY2, INTERNAL_NET)
df.to_csv(f'{MITRE_DIR}/mitre_test2_sgnn_noent.csv', index=False)

del model; gc.collect(); torch.cuda.empty_cache()
print('Done.')

In [ ]:
# ── Static GNN + Entropy ───────────────────────────────────────────────────
print('Running Static GNN + Entropy...')
model, threshold, _ = load_champion(
    StaticGNN_Identity_Entropy, 'StaticGNN_BiasOn_robust_Identity_entropy', df_champ_ent, DIR_ENT, DEVICE
)

df = extract_mitre_events(loader_t1_ent, DEVICE, model, threshold, False, id_to_ip_day1,
                          C2_SERVERS_DAY1, EXTERNAL_ATTACKERS_DAY1, INTERNAL_NET)
df.to_csv(f'{MITRE_DIR}/mitre_test1_sgnn_ent.csv', index=False)

df = extract_mitre_events(loader_t2_ent, DEVICE, model, threshold, False, id_to_ip_day2,
                          C2_SERVERS_DAY2, EXTERNAL_ATTACKERS_DAY2, INTERNAL_NET)
df.to_csv(f'{MITRE_DIR}/mitre_test2_sgnn_ent.csv', index=False)

del model; gc.collect(); torch.cuda.empty_cache()
print('Done.')

In [ ]:
# ── ST-GNN (noentropy) ─────────────────────────────────────────────────────
print('Running ST-GNN...')
model, threshold, _ = load_champion(
    ST_GNN_Identity, 'ST_GNN_BiasOn_robust_Identity_clone', df_champ_noent, DIR_NOENT, DEVICE
)

df = extract_mitre_events(loader_t1_noent, DEVICE, model, threshold, True, id_to_ip_day1,
                          C2_SERVERS_DAY1, EXTERNAL_ATTACKERS_DAY1, INTERNAL_NET)
df.to_csv(f'{MITRE_DIR}/mitre_test1_stgnn_noent.csv', index=False)

df = extract_mitre_events(loader_t2_noent, DEVICE, model, threshold, True, id_to_ip_day2,
                          C2_SERVERS_DAY2, EXTERNAL_ATTACKERS_DAY2, INTERNAL_NET)
df.to_csv(f'{MITRE_DIR}/mitre_test2_stgnn_noent.csv', index=False)

del model; gc.collect(); torch.cuda.empty_cache()
print('Done.')

In [ ]:
# ── E-GraphSAGE (noentropy) ───────────────────────────────────────────────
print('Running E-GraphSAGE...')
model, threshold, _ = load_champion(
    E_GraphSAGE, 'EGraphSAGE_BiasOn', df_champ_noent, DIR_NOENT, DEVICE
)

df = extract_mitre_events(loader_t1_noent, DEVICE, model, threshold, False, id_to_ip_day1,
                          C2_SERVERS_DAY1, EXTERNAL_ATTACKERS_DAY1, INTERNAL_NET)
df.to_csv(f'{MITRE_DIR}/mitre_test1_esage_noent.csv', index=False)

df = extract_mitre_events(loader_t2_noent, DEVICE, model, threshold, False, id_to_ip_day2,
                          C2_SERVERS_DAY2, EXTERNAL_ATTACKERS_DAY2, INTERNAL_NET)
df.to_csv(f'{MITRE_DIR}/mitre_test2_esage_noent.csv', index=False)
del model; gc.collect()
print('E-GraphSAGE done.')


In [ ]:
# ── EdgeGRU_NoX (noentropy) ───────────────────────────────────────────────
print('Running Edge GRU (NoX)...')
model, threshold, _ = load_champion(
    EdgeGRU_Baseline_NoX, 'EdgeGRU_NoX_BiasOn', df_champ_noent, DIR_NOENT, DEVICE
)

df = extract_mitre_events(loader_t1_noent, DEVICE, model, threshold, True, id_to_ip_day1,
                          C2_SERVERS_DAY1, EXTERNAL_ATTACKERS_DAY1, INTERNAL_NET)
df.to_csv(f'{MITRE_DIR}/mitre_test1_egru_nox.csv', index=False)

df = extract_mitre_events(loader_t2_noent, DEVICE, model, threshold, True, id_to_ip_day2,
                          C2_SERVERS_DAY2, EXTERNAL_ATTACKERS_DAY2, INTERNAL_NET)
df.to_csv(f'{MITRE_DIR}/mitre_test2_egru_nox.csv', index=False)
del model; gc.collect()
print('Edge GRU (NoX) done.')


# Section 2 — Tactic-level Comparison (Test 2)

In [ ]:
# ── Load all Test 2 CSVs ───────────────────────────────────────────────────
MODEL_KEYS = [
    ('egru_nox',   'Edge GRU'),
    ('egru_ent',   'Edge GRU + Entropy'),
    ('esage_noent','E-GraphSAGE'),
    ('sgnn_noent', 'Static GNN'),
    ('sgnn_ent',   'Static GNN + Entropy'),
    ('stgnn_noent','ST-GNN'),
    ('stgnn_ent',  'ST-GNN + Entropy'),
]

MODEL_ORDER = ['Edge GRU', 'Edge GRU + Entropy', 'E-GraphSAGE',
               'Static GNN', 'Static GNN + Entropy', 'ST-GNN', 'ST-GNN + Entropy']

TACTIC_ORDER = [
    'Initial Access', 'Command & Control',
    'Lateral Movement', 'Discovery', 'Internal Malicious Activity'
]

dfs_t2 = {}
for key, display in MODEL_KEYS:
    path = f'{MITRE_DIR}/mitre_test2_{key}.csv'
    dfs_t2[display] = pd.read_csv(path)
    print(f'{display}: {len(dfs_t2[display])} relevant flows')

In [ ]:
# ── metrics_per_tactic for each model ─────────────────────────────────────
# Total flows is the same for all models (same test2 dataset)
total_flows_t2 = count_total_flows(loader_t2_noent)  # noent and ent have same flow count

tactic_metrics = {}
for display, df in dfs_t2.items():
    tactic_metrics[display] = metrics_per_tactic(df, total_flows_t2, ignore_unknown=True)

In [ ]:
# ── Combined comparison table (Recall and F1 per tactic × model) ──────────
def pivot_metric(tactic_metrics_dict, metric_col, model_order, tactic_order):
    rows = []
    for model in model_order:
        df_m = tactic_metrics_dict.get(model, pd.DataFrame())
        for tactic in tactic_order:
            row_t = df_m[df_m['MITRE_Tactic'] == tactic]
            val = row_t[metric_col].iloc[0] if not row_t.empty else float('nan')
            rows.append({'Model': model, 'Tactic': tactic, metric_col: val})
    return pd.DataFrame(rows).pivot(index='Tactic', columns='Model', values=metric_col)

df_recall = pivot_metric(tactic_metrics, 'Recall (%)', MODEL_ORDER, TACTIC_ORDER)
df_f1     = pivot_metric(tactic_metrics, 'F1-Score (%)', MODEL_ORDER, TACTIC_ORDER)

print('Recall (%) per tactic × model')
display(df_recall.round(1))
print('\nF1-Score (%) per tactic × model')
display(df_f1.round(1))

In [ ]:
# ── Heatmap: Recall per tactic × model ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

for ax, (df_pivot, title) in zip(axes, [
    (df_recall, 'Recall (%)'),
    (df_f1,     'F1-Score (%)'),
]):
    df_plot = df_pivot[MODEL_ORDER].reindex(TACTIC_ORDER)
    sns.heatmap(
        df_plot, annot=True, fmt='.1f', cmap='YlGn',
        vmin=0, vmax=100, linewidths=0.5,
        cbar_kws={'label': title}, ax=ax
    )
    ax.set_title(f'{title} per MITRE Tactic', fontsize=12, weight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/mitre_tactic_heatmap.png', dpi=300)
plt.show()

In [ ]:
# ── Alert volume: TP and FP per model ─────────────────────────────────────
alert_rows = []
for display, df in dfs_t2.items():
    alert_rows.append({
        'Model': display,
        'TP': int(((df['y_real'] == 1) & (df['y_pred'] == 1)).sum()),
        'FP': int(((df['y_real'] == 0) & (df['y_pred'] == 1)).sum()),
        'FN': int(((df['y_real'] == 1) & (df['y_pred'] == 0)).sum()),
    })

df_alerts = pd.DataFrame(alert_rows)
df_alerts['Precision'] = (df_alerts['TP'] / (df_alerts['TP'] + df_alerts['FP'])).round(3)
df_alerts['Recall']    = (df_alerts['TP'] / (df_alerts['TP'] + df_alerts['FN'])).round(3)
print('Alert volume per model (Test 2):')
display(df_alerts.set_index('Model'))

# Chart
x = np.arange(len(df_alerts))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - w/2, df_alerts['TP'], w, label='True Positives',  color='#2ecc71')
ax.bar(x + w/2, df_alerts['FP'], w, label='False Positives', color='#e74c3c', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(df_alerts['Model'], rotation=25, ha='right', fontsize=9)
ax.set_title('Alert Volume per Model — Test 2', fontsize=12, weight='bold')
ax.set_ylabel('Flow count')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/mitre_alert_volume.png', dpi=300)
plt.show()

# Section 3 — Time-to-Detect Comparison (Test 2)

In [ ]:
# ── analyze_timeline for each model ───────────────────────────────────────
ttd_per_model = {}
for display, df in dfs_t2.items():
    df_gt = df[df['y_real'] == 1].copy()
    df_tp = df[(df['y_real'] == 1) & (df['y_pred'] == 1)].copy()
    if df_tp.empty:
        print(f'{display}: no TPs found, skipping timeline.')
        continue
    ttd_per_model[display] = analyze_timeline(df_gt, df_tp, time_window_sec=30)

In [ ]:
# ── Cross-model TTD table ─────────────────────────────────────────────────
ttd_rows = []
for tactic in TACTIC_ORDER:
    row = {'Tactic': tactic}
    for model in MODEL_ORDER:
        df_ttd = ttd_per_model.get(model, pd.DataFrame())
        match = df_ttd[df_ttd['MITRE_Tactic_Base'] == tactic]
        row[model] = round(match['TTD_Minutes'].iloc[0], 1) if not match.empty else float('nan')
    ttd_rows.append(row)

df_ttd_compare = pd.DataFrame(ttd_rows).set_index('Tactic')
print('Time-to-Detect (minutes) per tactic × model  [NaN = tactic not detected]')
display(df_ttd_compare)

In [ ]:
# ── Window consistency: % of attack windows detected per tactic × model ───
consist_rows = []
for tactic in TACTIC_ORDER:
    row = {'Tactic': tactic}
    for model in MODEL_ORDER:
        df_ttd = ttd_per_model.get(model, pd.DataFrame())
        match = df_ttd[df_ttd['MITRE_Tactic_Base'] == tactic]
        row[model] = round(match['Window_Consistency_Pct'].iloc[0], 1) if not match.empty else float('nan')
    consist_rows.append(row)

df_consistency = pd.DataFrame(consist_rows).set_index('Tactic')

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    df_consistency[MODEL_ORDER], annot=True, fmt='.1f', cmap='Blues',
    vmin=0, vmax=100, linewidths=0.5,
    cbar_kws={'label': 'Window Consistency (%)'}, ax=ax
)
ax.set_title('Detection Consistency per Tactic (% of attack windows detected)', fontsize=11, weight='bold')
ax.set_xlabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/mitre_window_consistency.png', dpi=300)
plt.show()

# Section 4 — Deep Dive: ST-GNN + Entropy (Test 2)

Full temporal analysis for the best model. Loads from the saved CSV — no rerun needed.

In [ ]:
df_stgnn_ent = pd.read_csv(f'{MITRE_DIR}/mitre_test2_stgnn_ent.csv')

df_gt_t2  = df_stgnn_ent[df_stgnn_ent['y_real'] == 1].copy()
df_tp_t2  = df_stgnn_ent[(df_stgnn_ent['y_real'] == 1) & (df_stgnn_ent['y_pred'] == 1)].copy()
df_fp_t2  = df_stgnn_ent[(df_stgnn_ent['y_real'] == 0) & (df_stgnn_ent['y_pred'] == 1)].copy()

print(f'GT flows: {len(df_gt_t2)} | TP: {len(df_tp_t2)} | FP: {len(df_fp_t2)}')

In [ ]:
# Temporal metrics summary
calculate_full_temporal_metrics(df_gt_t2, df_tp_t2, df_fp_t2, ignore_unknown=True)

In [ ]:
# Early warning before lateral movement
ew = analyze_early_warning(df_tp_t2, time_window_sec=30)
print('Early Warning Analysis:')
for k, v in ew.items():
    print(f'  {k}: {v:.2f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# Lateral movement pairs
df_lm = analyze_all_lateral_movements_pairs(df_tp_t2, time_window_sec=30)
display(df_lm)

In [ ]:
# MTTD per tactic
df_mttd = calculate_mttd_metrics(df_gt_t2, df_tp_t2, time_window_sec=30, max_gap_windows=5)
display(df_mttd)

In [ ]:
# MTTD bar chart
df_plot = df_mttd.sort_values('Real_Start_Minute').copy()
errores = [np.zeros(len(df_plot)), df_plot['Std_TTD_Minutes'].values]

fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(len(df_plot))
ax.barh(y_pos, df_plot['MTTD_Minutes'], xerr=errores, capsize=4,
        color='#1e8449', alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['MITRE_Tactic_Base'], fontsize=10)
ax.set_xlabel('Mean Time-to-Detect (minutes)')
ax.set_title('MTTD per MITRE Tactic — ST-GNN + Entropy', fontsize=12, weight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/mttd_stgnn_entropy.png', dpi=300)
plt.show()

In [ ]:
# FP analysis — which tactics generate the most false alarms
print('False Positive breakdown by port category:')
display(df_fp_t2['Port_Category'].value_counts().rename('FP count').to_frame())

print('\nTop source IPs generating false alarms:')
display(df_fp_t2['Source_IP'].value_counts().head(10).rename('FP count').to_frame())